# Employee Performance Factor Analysis

## Project Overview
Analyze performance review data to understand which factors relate most strongly to high and low performance ratings.

## Learning Objectives
- Identify key drivers of employee performance
- Analyze performance distributions across departments, levels, and tenure
- Compare factor profiles of top vs bottom performers
- Quantify correlations between engagement, training, and performance

## Problem Statement
HR wants to understand what distinguishes high performers from low performers beyond intuition. They need data to guide coaching, development, and talent management decisions.

## Why This Project Matters
Understanding performance drivers enables targeted development programs, better hiring criteria, and evidence-based promotion decisions.

## Dataset Overview
Synthetic performance dataset: ~4,000 employees with ratings, engagement, training hours, tenure, manager ratings, and work metrics.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 4000
departments = np.random.choice(['Engineering', 'Sales', 'Marketing', 'Finance', 'Operations', 'Product'],
                                 n, p=[0.24, 0.20, 0.14, 0.13, 0.16, 0.13])
levels = np.random.choice(['Junior', 'Mid', 'Senior', 'Lead', 'Manager'], n, p=[0.24, 0.30, 0.25, 0.13, 0.08])
tenure = np.clip(np.random.exponential(4, n), 0.5, 18).round(1)
engagement = np.clip(np.random.normal(3.6, 0.8, n), 1.0, 5.0).round(1)
training_hours = np.clip(np.random.lognormal(2.5, 0.8, n), 0, 120).round(0)
projects_completed = np.clip(np.random.poisson(5, n), 0, 20)
manager_support = np.clip(np.random.normal(3.5, 0.9, n), 1.0, 5.0).round(1)
work_life_balance = np.clip(np.random.normal(3.3, 0.8, n), 1.0, 5.0).round(1)
remote_pct = np.random.choice([0, 25, 50, 75, 100], n, p=[0.20, 0.15, 0.25, 0.20, 0.20])

# Performance rating driven by factors
latent = (0.25 * engagement + 0.15 * manager_support + 0.10 * (training_hours / 30) +
          0.15 * (projects_completed / 5) + 0.10 * work_life_balance +
          0.05 * np.minimum(tenure, 8) / 8 + np.random.normal(0, 0.3, n))
perf_score = np.clip(latent, 1, 5).round(1)
perf_rating = pd.cut(perf_score, bins=[0, 2.0, 3.0, 3.8, 4.5, 5.1],
                      labels=['Needs Improvement', 'Developing', 'Meets Expectations', 'Exceeds', 'Outstanding'])

df = pd.DataFrame({
    'EmployeeID': [f'E{i:05d}' for i in range(n)],
    'Department': departments, 'Level': levels,
    'TenureYears': tenure, 'EngagementScore': engagement,
    'TrainingHours': training_hours, 'ProjectsCompleted': projects_completed,
    'ManagerSupport': manager_support, 'WorkLifeBalance': work_life_balance,
    'RemotePct': remote_pct, 'PerfScore': perf_score,
    'PerfRating': perf_rating,
})

print(f'Dataset: {df.shape}')
print(f'\nPerformance Rating distribution:\n{df["PerfRating"].value_counts().sort_index()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nNumeric summary:')
print(df.describe().round(2))

## Performance Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['PerfScore'].hist(bins=30, ax=axes[0], edgecolor='black', color='steelblue', alpha=0.7)
axes[0].set_title('Performance Score Distribution')

rating_order = ['Needs Improvement', 'Developing', 'Meets Expectations', 'Exceeds', 'Outstanding']
df['PerfRating'].value_counts().reindex(rating_order).plot.bar(ax=axes[1], edgecolor='black', color='coral')
axes[1].set_title('Performance Rating Distribution')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'perf_distribution.png'), dpi=100, bbox_inches='tight')
plt.show()

## Factor Correlations

In [ ]:
num_cols = ['EngagementScore', 'TrainingHours', 'ProjectsCompleted',
           'ManagerSupport', 'WorkLifeBalance', 'TenureYears', 'RemotePct', 'PerfScore']
corr = df[num_cols].corr().round(3)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Factor Correlation Matrix')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'correlation_matrix.png'), dpi=100, bbox_inches='tight')
plt.show()

print('\nCorrelation with PerfScore:')
print(corr['PerfScore'].drop('PerfScore').sort_values(ascending=False))

## Top vs Bottom Performer Profiles

In [ ]:
top = df[df['PerfRating'].isin(['Exceeds', 'Outstanding'])]
bottom = df[df['PerfRating'].isin(['Needs Improvement', 'Developing'])]
factors = ['EngagementScore', 'TrainingHours', 'ProjectsCompleted', 'ManagerSupport', 'WorkLifeBalance', 'TenureYears']

comparison = pd.DataFrame({
    'Top Performers': top[factors].mean(),
    'Bottom Performers': bottom[factors].mean(),
    'Diff': top[factors].mean() - bottom[factors].mean(),
}).round(2)
print('Top vs Bottom Performer Profile:')
print(comparison)

fig, ax = plt.subplots(figsize=(10, 6))
comparison[['Top Performers', 'Bottom Performers']].plot.barh(ax=ax, edgecolor='black')
ax.set_title('Factor Averages: Top vs Bottom Performers')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top_vs_bottom.png'), dpi=100, bbox_inches='tight')
plt.show()

## Performance by Department and Level

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dept_perf = df.groupby('Department')['PerfScore'].mean().sort_values()
dept_perf.plot.barh(ax=axes[0], edgecolor='black', color='teal')
axes[0].set_title('Avg Performance Score by Department')

level_order = ['Junior', 'Mid', 'Senior', 'Lead', 'Manager']
level_perf = df.groupby('Level')['PerfScore'].mean().reindex(level_order)
level_perf.plot.bar(ax=axes[1], edgecolor='black', color='goldenrod')
axes[1].set_title('Avg Performance Score by Level')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'dept_level_perf.png'), dpi=100, bbox_inches='tight')
plt.show()

## Engagement × Manager Support Interaction

In [ ]:
df['EngBucket'] = pd.cut(df['EngagementScore'], bins=[0, 2.5, 3.5, 4.5, 5.1],
                          labels=['Low', 'Medium', 'High', 'Very High'])
df['MgrBucket'] = pd.cut(df['ManagerSupport'], bins=[0, 2.5, 3.5, 4.5, 5.1],
                          labels=['Low', 'Medium', 'High', 'Very High'])
cross = df.groupby(['EngBucket', 'MgrBucket'])['PerfScore'].mean().unstack().round(2)
print('Avg Performance: Engagement × Manager Support')
print(cross)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cross, annot=True, fmt='.2f', cmap='YlGnBu', ax=ax)
ax.set_title('Avg Performance: Engagement × Manager Support')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'engagement_manager.png'), dpi=100, bbox_inches='tight')
plt.show()

## Training Hours Impact

In [ ]:
df['TrainingBand'] = pd.cut(df['TrainingHours'], bins=[0, 10, 25, 50, 999],
                             labels=['<10h', '10-25h', '25-50h', '50h+'])
train_perf = df.groupby('TrainingBand').agg(
    employees=('EmployeeID', 'count'),
    avg_perf=('PerfScore', 'mean'),
    pct_top=('PerfRating', lambda x: (x.isin(['Exceeds', 'Outstanding'])).mean()),
).round(3)
print('Performance by Training Hours:')
print(train_perf)

## Interpretation and Business Insight
- **Engagement** is the strongest single predictor of performance — investing in engagement drives results
- **Manager support** has a strong independent effect and amplifies engagement — good managers multiply performance
- **Training hours** show positive returns up to ~50h, with diminishing returns beyond that
- **Projects completed** correlates with performance — giving people meaningful work matters
- **Work-life balance** positively correlates — overworked employees don't outperform, they burn out
- The **interaction** between engagement and manager support is powerful: low engagement + low manager support = worst outcomes

## Limitations
- Synthetic data with pre-programmed factor weights
- No causal inference — correlations don't prove causation
- Performance ratings are often biased by rating inflation and manager calibration
- No 360-degree feedback or peer review data
- Missing important factors: role fit, team dynamics, organizational culture

## How to Improve This Project
- Use real anonymized performance data
- Add causal analysis (instrumental variables, natural experiments)
- Include 360-degree feedback and peer ratings
- Track performance trajectory over multiple review cycles
- Add manager-level random effects analysis

## Production Considerations
- Quarterly performance driver dashboards for HR business partners
- Manager coaching tools highlighting team factor gaps
- Automated identification of high-potential employees with underperforming factors
- Integration with engagement survey platforms

## Common Mistakes
- Assuming correlation = causation (engagement predicts performance, but high performance also drives engagement)
- Ignoring manager calibration differences across teams
- Using single-point ratings instead of trajectory analysis
- Not controlling for role complexity when comparing performance across levels

## Mini Challenge / Exercises
1. Which single factor, if improved by 1 unit for all employees, would yield the largest average performance increase?
2. Identify employees with high engagement but low performance — what other factors explain this?
3. Calculate the % of performance variance explained by engagement alone vs all factors combined.
4. Which department has the largest gap between top and bottom performers?

## Final Summary / Key Takeaways
- Performance is driven by a combination of engagement, manager support, meaningful work, and work-life balance
- No single factor is sufficient — high performance requires multiple enabling conditions
- Manager effectiveness is an underappreciated performance multiplier
- Training has positive but diminishing returns — quality matters more than quantity
- Understanding performance drivers enables targeted interventions over generic programs